In [1]:
import pandas as pd

df = pd.read_parquet("data/clustered_data_depth1.parquet")
best = df[df["cluster"] == 1]

print(f"Total rows: {len(best)}")
print(f"Unique tickers: {best['ticker'].nunique()}")
print(f"Avg rows per ticker: {len(best) / best['ticker'].nunique():.1f}")

print("\n=== Sector breakdown ===")
print(best["sector"].value_counts())
print("\nAs % of cluster:")
print((best["sector"].value_counts(normalize=True) * 100).round(1))

print("\n=== Date range ===")
print(f"{best['date'].min()} to {best['date'].max()}")

best = best.copy()
best["year"] = pd.to_datetime(best["date"]).dt.year
print("\nRows by year:")
print(best["year"].value_counts().sort_index())

print("\n=== Ticker concentration ===")
top10 = best["ticker"].value_counts().head(10)
print(top10)
print(f"\nTop 10 tickers as % of cluster: {top10.sum() / len(best):.1%}")

Total rows: 17843
Unique tickers: 3082
Avg rows per ticker: 5.8

=== Sector breakdown ===
sector
Financial Services        4947
Industrials               2740
Consumer Cyclical         2017
Technology                1584
Healthcare                1484
Energy                    1056
Real Estate                977
Basic Materials            912
Communication Services     670
Consumer Defensive         564
Unknown                    455
Utilities                  437
Name: count, dtype: int64

As % of cluster:
sector
Financial Services        27.7
Industrials               15.4
Consumer Cyclical         11.3
Technology                 8.9
Healthcare                 8.3
Energy                     5.9
Real Estate                5.5
Basic Materials            5.1
Communication Services     3.8
Consumer Defensive         3.2
Unknown                    2.6
Utilities                  2.4
Name: proportion, dtype: float64

=== Date range ===
2018-12-21 00:00:00 to 2020-06-26 00:00:00

Rows by yea

In [6]:
import pandas as pd
import numpy as np

# ----------------------------
# Load data
# ----------------------------
all_df = pd.read_parquet("data/clustered_data.parquet")
best_df = pd.read_parquet("data/best_cluster_depth2.parquet")

print(f"All rows: {len(all_df):,}")
print(f"Best rows: {len(best_df):,}")

# ----------------------------
# Create comparison dataset
# ----------------------------
rest_df = all_df.loc[~all_df.index.isin(best_df.index)]

print(f"Rest rows: {len(rest_df):,}")

# ----------------------------
# Select numeric features
# ----------------------------
exclude = {
    "future_5y_return",
    "cluster",
    "ticker",
    "date",
    "sector",
    "year",
}

feature_cols = [
    c for c in all_df.columns
    if c not in exclude
    and pd.api.types.is_numeric_dtype(all_df[c])
]

print(f"Comparing {len(feature_cols)} numeric features")

# ----------------------------
# Compare feature means
# ----------------------------
comparison = pd.DataFrame({
    "best_mean": best_df[feature_cols].mean(),
    "rest_mean": rest_df[feature_cols].mean(),
})

comparison["difference"] = (
    comparison["best_mean"] -
    comparison["rest_mean"]
)

# Standardized difference
std = all_df[feature_cols].std().replace(0, np.nan)

comparison["effect_size"] = (
    comparison["difference"] / std
)

comparison = comparison.sort_values(
    "effect_size",
    key=lambda s: s.abs(),
    ascending=False
)

# nicer formatting
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

print("\nTop distinguishing features\n")
print(comparison.head(30))

# ----------------------------
# Save results
# ----------------------------
comparison.to_csv(
    "data/cluster_feature_comparison.csv"
)

print("\nSaved to data/cluster_feature_comparison.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/clustered_data.parquet'

In [2]:
df["year"] = pd.to_datetime(df["date"]).dt.year
cluster1 = df[df["cluster"] == 1]

for yr in [2018, 2020]:
    subset = cluster1[cluster1["year"] == yr]
    print(f"\nYear {yr}: n={len(subset)}")
    print(f"  Mean future_5y_return: {subset['future_5y_return'].mean():.2%}")
    print(f"  Median future_5y_return: {subset['future_5y_return'].median():.2%}")


Year 2018: n=1339
  Mean future_5y_return: 110.02%
  Median future_5y_return: 60.97%

Year 2020: n=16504
  Mean future_5y_return: 129.04%
  Median future_5y_return: 74.23%
